In [13]:
import glob
import re
import pandas as pd
import shutil
files = glob.glob("../data/K*")

In [14]:
f_name = []
for f in files:
    p = 'data/(K\d{6}).TXT'
    f_name.append(re.search(p,f).group(1))

In [15]:
f_name

['K200901',
 'K200929',
 'K210509',
 'K210521',
 'K200726',
 'K210119',
 'K201014',
 'K210131',
 'K210125',
 'K201028',
 'K201202',
 'K201216',
 'K210327',
 'K210331',
 'K210325',
 'K201228',
 'K210319',
 'K201214',
 'K210127',
 'K201002',
 'K201016',
 'K210523',
 'K200730',
 'K200724',
 'K200718',
 'K200917',
 'K200903',
 'K200902',
 'K200916',
 'K200719',
 'K200725',
 'K200731',
 'K210522',
 'K201017',
 'K201003',
 'K210126',
 'K201215',
 'K210318',
 'K201201',
 'K201229',
 'K210324',
 'K210330',
 'K210308',
 'K201205',
 'K201211',
 'K210320',
 'K201007',
 'K201013',
 'K210122',
 'K200709',
 'K210526',
 'K200721',
 'K200912',
 'K200906',
 'K200907',
 'K200913',
 'K200720',
 'K210527',
 'K200708',
 'K210123',
 'K201012',
 'K201006',
 'K210321',
 'K201210',
 'K201204',
 'K210309',
 'K201212',
 'K201206',
 'K210323',
 'K201010',
 'K201004',
 'K210109',
 'K210121',
 'K210519',
 'K200722',
 'K210531',
 'K210525',
 'K200905',
 'K200911',
 'K200910',
 'K200904',
 'K210524',
 'K210530',
 'K2

In [16]:
def mkcsv(FILE_NAME):
    with open('../data/' + FILE_NAME + '.TXT', encoding='shift-jis') as f:
        data = f.readlines()
    racers = []
    race = []
    place = []
    for i in data:
        if  re.match('^\s{3}第.+日', i):
            place.append(i.replace('\u3000','').replace('\n','').replace('/','_').replace('_ ', '_'))
        elif re.match('^\s+\d+R.+H', i):
            race.append(i.replace('\u3000','').replace('\n',''))
        elif re.match('^\s{2}0[1-9]\s', i):
            racers.append(i.replace('\u3000','').replace('\n',''))

    pattern_place = re.compile('^.+(\d{4}_.+\d+)\s.+ボートレース([^0-9A-Z\s-]+)')
    pl = [re.match(pattern_place, i).groups() for i in place]

    pattern_race = re.compile('^\s+(\d+R).+\s{2}([^0-9A-Z\s-]+)\s{2}風\s{2}([^0-9A-Z\s]+).+(\d+)m.+波.+(\d+)cm')
    rc = [re.match(pattern_race, i).groups() for i in race]

    pattern_racers = re.compile('^\s{2}0([1-6])\s{2}([1-6])\s(\d{4})\s([^0-9]+)\s(\d+)\s+(\d+)\s{2}(\d{1}.\d{2})\s{3}(\d)\s{3}[\sF](\d.\d{2})\s+')
    rs = [re.match(pattern_racers, i).groups() for i in racers]

    column_rs = ['着','艇番','選手登番','選手名','モーター','ボート','展示','進入','スタートタイミング']
    df_rs = pd.DataFrame(rs, columns=column_rs)

    column_rc = ['R','天気','風向','風量','波']
    df_rc = pd.DataFrame(rc, columns=column_rc)

    column_pl = ['日時','場所']
    df_pl = pd.DataFrame(pl, columns=column_pl)
    df_rc['場所'] = ''
    df_rc['日時'] = ''
    m = -1
    for i in range(df_rc.shape[0]):
        n = df_rc['R'][i]
        if n == '1R':
            m = m + 1
            df_rc['場所'][i] = df_pl['場所'][m]
            df_rc['日時'][i] = df_pl['日時'][m]
        else:
            df_rc['場所'][i] = df_pl['場所'][m]
            df_rc['日時'][i] = df_pl['日時'][m]

    m = -1
    df_rs['場所'] = ''
    for c in df_rc.columns:
        df_rs[c] = ''

    for i in range(df_rs.shape[0]):
        n = int(df_rs['着'][i])
        if n == 1  :
            m = m + 1
            for c in df_rc.columns:
                df_rs[c][i] = df_rc[c][m]
        else:
            for c in df_rc.columns:
                df_rs[c][i] = df_rc[c][m]
    place_code = {'桐生':'01','戸田':'02','江戸川':'03','平和島':'04','多摩川':'05','浜名湖':'06','蒲郡':'07','常滑':'08','津':'09','三国':'10','びわこ':'11','住之江':'12','尼崎':'13','鳴門':'14','丸亀':'15','児島':'16','宮島':'17','徳山':'18','下関':'19','若松':'20','芦屋':'21','福岡':'22','唐津':'23','大村':'24'}

    df_rs['場所'] = df_rs['場所'].map(place_code)
    df_rs['RaceID'] = df_rs['日時'] + '_' + df_rs['場所'] + '_' + df_rs['R']

    df_rs.to_csv('../csv/' + FILE_NAME +  '.csv')
    shutil.move('../data/' + FILE_NAME + '.TXT', '../done/' + FILE_NAME + '.TXT')


In [17]:
import tqdm
for f in tqdm(f_name):
    mkcsv(f)